# Create weighingGUI Spreadsheet daily

In [1]:
from scripts.conf_file_finding import try_find_conf_file
try_find_conf_file()

Local configuration file found !!, no need to run the configuration (unless configuration has changed)


In [98]:
import datajoint as dj
import pandas as pd
import time
from zoneinfo import ZoneInfo
import datetime
import pathlib
import numpy as np
import time
import base64
import shutil
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.drawing.image import Image

time.sleep(1)

import u19_pipeline.alert_system.water_weigh_alert.water_weigh_alert as wwa
import u19_pipeline.scheduler as scheduler
import u19_pipeline.acquisition as acquisition
import u19_pipeline.behavior as behavior
import u19_pipeline.subject as subject
import u19_pipeline.lab as lab


DJ_CUSTOM_VARIABLES_FILENAME = 'DJCustomVariables.csv'
SLACK_WEBHOOK_FILENAME = 'SlackChannels.csv'
USER_SLACK_FILENAME = 'UserSlack.csv'
RIG_STATUS_FILENAME = 'RigStatusTable.csv'
DAY_SCHEDULE_FILENAME = 'ScheduleDay.csv'
PAST_SESSION_PERFORMANCE_FILENAME = 'PastSessions.csv'

WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME_TEMPLATE = 'Weighing_GUI_Replacement_SpreadSheet_Template.xlsx'
WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME = 'Weighing_GUI_Replacement_SpreadSheet.xlsx'



MAX_SESSIONS_HISTORY = 75

In [99]:
def cast_choice(choice_array):

    new_array = choice_array[0].copy()
    new_array[new_array>2] = 127

    new_array = np.array(new_array, dtype=np.uint8)

    return new_array


In [100]:
def encode_webhook(webhook):

    return (base64.b64encode(webhook.encode('utf-8'))).decode('utf-8')
   

In [101]:
conf = dj.config

nodb_virmen_backup_dir = pathlib.Path(pathlib.Path(conf['custom']['root_data_dir'][0]).parent.parent,'Shared','NoDBVirmenBackup')

DJ_CUSTOM_VARIABLES_FILENAME = pathlib.Path(nodb_virmen_backup_dir,DJ_CUSTOM_VARIABLES_FILENAME).as_posix()
SLACK_WEBHOOK_FILENAME = pathlib.Path(nodb_virmen_backup_dir,SLACK_WEBHOOK_FILENAME).as_posix()
USER_SLACK_FILENAME = pathlib.Path(nodb_virmen_backup_dir,USER_SLACK_FILENAME).as_posix()
RIG_STATUS_FILENAME = pathlib.Path(nodb_virmen_backup_dir,RIG_STATUS_FILENAME).as_posix()
DAY_SCHEDULE_FILENAME = pathlib.Path(nodb_virmen_backup_dir,DAY_SCHEDULE_FILENAME).as_posix()
PAST_SESSION_PERFORMANCE_FILENAME = pathlib.Path(nodb_virmen_backup_dir,PAST_SESSION_PERFORMANCE_FILENAME).as_posix()

WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME = pathlib.Path(nodb_virmen_backup_dir,\
                                                            WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME).as_posix()

WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME_TEMPLATE = pathlib.Path(nodb_virmen_backup_dir,\
                                                            WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME_TEMPLATE).as_posix()




In [6]:
pd.DataFrame(lab.DjCustomVariables.fetch(as_dict=True)).to_csv(DJ_CUSTOM_VARIABLES_FILENAME)


In [7]:
slack_webhooks = pd.DataFrame(lab.SlackWebhooks.fetch(as_dict=True))
slack_webhooks['webhook_url'] = slack_webhooks['webhook_url'].astype(str)

slack_webhooks['webhook_url'] = slack_webhooks['webhook_url'].apply(encode_webhook)
slack_webhooks.to_csv(SLACK_WEBHOOK_FILENAME, index=False)

In [8]:
user_data = pd.DataFrame(lab.User.fetch('user_id', 'slack', 'tech_responsibility', 'slack_webhook',as_dict=True))
user_data['slack_webhook'] = user_data['slack_webhook'].astype(str)
user_data['slack_webhook'] = user_data['slack_webhook'].fillna('')

user_data['slack_webhook'] = user_data['slack_webhook'].apply(encode_webhook)
user_data.to_csv(USER_SLACK_FILENAME, index=False)

In [9]:
user_data

,user_id,slack,tech_responsibility,slack_webhook
0,aarusso,Abby Russo,yes,aHR0cHM6Ly9ob29rcy5zbGFjay5jb20vc2VydmljZXMvVD...
1,ad9966,U08J0JCUEFP,N/A,
2,alvaros,UUD6UMWCS,yes,aHR0cHM6Ly9ob29rcy5zbGFjay5jb20vc2VydmljZXMvVD...
3,ao8210,U0A7SMLNM1D,yes,Tm9uZQ==
4,apv2,UDQLJ5MKM,yes,Tm9uZQ==
5,ariordan,Alex Riordan,yes,aHR0cHM6Ly9ob29rcy5zbGFjay5jb20vc2VydmljZXMvVD...
6,baptista,Scott Baptista,N/A,
7,bm9751,U06UUR8M9A6,yes,Tm9uZQ==
8,ct5868,U06GN5TBX46,no,Tm9uZQ==
9,de4003,U09KSJAVACD,yes,Tm9uZQ==


In [10]:
df_rig_status = pd.DataFrame(scheduler.RigStatus.fetch('location', 'input_output_name','current_status',as_dict=True))
df_rig_status.to_csv(RIG_STATUS_FILENAME, index=False)

In [11]:
schedule_query = dict()
schedule_query['date'] = datetime.date.today() + datetime.timedelta(days=1)

subject_query = 'subject_fullname is not null'

day_schedule = pd.DataFrame((scheduler.Schedule * scheduler.TrainingProfile * subject.Subject.proj(subject_user_id='user_id') & schedule_query & subject_query).fetch(as_dict=True))
day_schedule = day_schedule.drop(columns='user_id')
day_schedule = day_schedule.rename(columns={'subject_user_id':'user_id'})


day_schedule.to_csv(DAY_SCHEDULE_FILENAME, index=False)

In [12]:
day_schedule

,date,location,timeslot,training_profile_id,subject_fullname,recording_profile_id,input_output_profile_id,experimenters_instructions,level,sublevel,date_created,date_last_used,training_profile_name,training_profile_description,training_profile_variables,deprecated,user_id
0,2026-03-06,165A-miniVR-T-1,1,145,efonseca_ef932_act131,1,5,"Do not modify DV coordinates, unless too obvio...",0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
1,2026-03-06,165A-miniVR-T-1,2,154,efonseca_ef717_sta531,1,5,Do NOT change DV (up-down) coordinates.,0,0,2026-01-21,2026-01-21,LSTT_Stationary_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
2,2026-03-06,165A-miniVR-T-1,3,145,efonseca_ef730_act134,1,5,Do not change DV coordinates,0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
3,2026-03-06,165A-miniVR-T-2,1,145,efonseca_ef512_act132,1,5,"Do not modify DV coordinates, unless too obvio...",0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
4,2026-03-06,165A-miniVR-T-2,2,154,efonseca_ef718_sta532,1,5,Do NOT change DV (up-down) coordinates. Thanks!,0,0,2026-01-21,2026-01-21,LSTT_Stationary_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
5,2026-03-06,165A-miniVR-T-2,3,145,efonseca_ef732_act701,1,5,Do not change DV coordinates. Thanks! :D,0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
6,2026-03-06,165A-miniVR-T-3,1,145,efonseca_ef505_act133,1,5,Do not change DV (up-down) coordinates. Thanks,0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
7,2026-03-06,165A-miniVR-T-3,2,152,efonseca_ef721_pas331,1,5,Do NOT change DV (up-down) coordinates. Thanks!,0,0,2025-10-09,2025-10-09,LSTT_Passive_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
8,2026-03-06,165A-miniVR-T-3,3,152,efonseca_ef722_pas332,1,5,Do not change DV coordinates. Thanks!,0,0,2025-10-09,2025-10-09,LSTT_Passive_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
9,2026-03-06,165A-miniVR-T-5,5,131,jk8386_jk112,1,2,"LASER ON\ndoric file: ""...triggered...50hz...""...",0,0,2025-04-14,2025-04-14,jesse_chronic_spatfreq_prot,Jesse's spatfreq task for optogenetics across ...,"{""rewardFactor"": [[1.0, 1.0, 1.0, 1.0, 1.0, 1....",0,jk8386


In [13]:
all_subjects_schedule = "', '".join(day_schedule['subject_fullname'])
all_subjects_schedule = "subject_fullname in ('" +all_subjects_schedule+ "')"
all_subjects_schedule

"subject_fullname in ('efonseca_ef932_act131', 'efonseca_ef717_sta531', 'efonseca_ef730_act134', 'efonseca_ef512_act132', 'efonseca_ef718_sta532', 'efonseca_ef732_act701', 'efonseca_ef505_act133', 'efonseca_ef721_pas331', 'efonseca_ef722_pas332', 'jk8386_jk112', 'jk8386_jk113', 'jounhong_DRD1cre_783', 'jounhong_DRD1cre_784', 'jounhong_DRD1cre_785', 'testuser_ef932_act131_T00', 'jeremyjc_j100', 'jeremyjc_j105', 'jeremyjc_j109', 'jeremyjc_j099', 'jeremyjc_j107', 'jeremyjc_j103', 'jyanar_ya066', 'jyanar_ya069', 'jyanar_ya070', 'jyanar_ya067', 'jyanar_ya065', 'jyanar_ya068', 'jyanar_ya071', 'jyanar_ya061', 'jyanar_ya063', 'jyanar_ya062', 'jyanar_ya064')"

In [14]:
ss = pd.DataFrame((behavior.TowersSession & all_subjects_schedule).fetch('KEY', order_by='subject_fullname, session_date desc, session_number desc', as_dict=True))
ss['session_date'] = ss['session_date'].astype(str)
ss['num_sessions'] = ss.groupby(['subject_fullname'])['subject_fullname'].rank(method='first')
ss['num_sessions'] = ss['num_sessions'].astype(int)
ss = ss.loc[ss['num_sessions'] <= MAX_SESSIONS_HISTORY, :]

#ss2 = ss.copy()

max_value_indices = ss.groupby('subject_fullname')['num_sessions'].idxmax()
ss = ss.loc[max_value_indices]
ss = ss.reset_index(drop=True)
ss['query'] = "(subject_fullname='" + ss['subject_fullname'] + "' and session_date >= '" + ss['session_date'] + "')"

block_query = ' OR '.join(ss['query'])
ss

,subject_fullname,session_date,session_number,num_sessions,query
0,efonseca_ef505_act133,2026-01-15,0,37,(subject_fullname='efonseca_ef505_act133' and ...
1,efonseca_ef512_act132,2026-01-10,0,43,(subject_fullname='efonseca_ef512_act132' and ...
2,efonseca_ef717_sta531,2026-01-29,2,24,(subject_fullname='efonseca_ef717_sta531' and ...
3,efonseca_ef718_sta532,2026-01-29,1,27,(subject_fullname='efonseca_ef718_sta532' and ...
4,efonseca_ef721_pas331,2026-01-29,0,25,(subject_fullname='efonseca_ef721_pas331' and ...
5,efonseca_ef722_pas332,2026-02-02,0,15,(subject_fullname='efonseca_ef722_pas332' and ...
6,efonseca_ef730_act134,2026-02-03,0,21,(subject_fullname='efonseca_ef730_act134' and ...
7,efonseca_ef732_act701,2026-02-03,0,20,(subject_fullname='efonseca_ef732_act701' and ...
8,efonseca_ef932_act131,2026-01-10,0,43,(subject_fullname='efonseca_ef932_act131' and ...
9,jeremyjc_j099,2025-10-20,0,75,(subject_fullname='jeremyjc_j099' and session_...


In [15]:
sstable=(acquisition.SessionStarted.proj('local_path_behavior_file', 'session_location'))
stable = (acquisition.Session).proj(stimulusBank='stimulus_bank')
tstable = (behavior.TowersSession.proj(trialType='rewarded_side',choice='chosen_side', stimulusSet='stimulus_set'))  
tbtable = (behavior.TowersBlock.proj('first_trial','n_trials', 'sublevel', mazeID='level', mainMazeID='main_level', easyBlockFlag='easy_block',\
                                     duration='block_duration', rewardMil='reward_mil', medianTrialDur='trial_duration_median', start='block_start_time'))  
table_fetch = sstable * stable* tstable * tbtable

allblocks = pd.DataFrame((table_fetch & block_query).fetch(order_by='subject_fullname, session_date desc, session_number desc, block desc',as_dict=True))
allblocks['from_DB'] = 1
allblocks['session_date'] = allblocks['session_date'].astype(str)

allblocks['sublevel'] = allblocks['sublevel'].astype('Int64')
allblocks['choice'] = allblocks['choice'].apply(cast_choice)
allblocks['trialType'] = allblocks['trialType'].apply(cast_choice)

#allblocks['session_date'] = allblocks['session_date'].astype(str)
#allblocks['num_blocks'] = allblocks.groupby(['subject_fullname'])['subject_fullname'].rank(method='first')

#all_sessions_blocks = allblocks.drop_duplicates(subset=['subject_fullname','session_date','session_number']).copy()
#all_sessions_blocks['num_session_blocks'] = all_sessions_blocks.groupby(['subject_fullname'])['subject_fullname'].rank(method='first')
#all_sessions_blocks2 = all_sessions_blocks.copy()
#max_value_indices = all_sessions_blocks.groupby('subject_fullname')['num_session_blocks'].idxmax()
#all_sessions_blocks = all_sessions_blocks.loc[max_value_indices]



 

In [16]:

allblocks.to_csv(PAST_SESSION_PERFORMANCE_FILENAME, index=False)
allblocks

,subject_fullname,session_date,session_number,block,session_location,local_path_behavior_file,stimulusBank,stimulusSet,trialType,choice,...,mazeID,sublevel,n_trials,first_trial,duration,start,rewardMil,easyBlockFlag,medianTrialDur,from_DB
0,efonseca_ef505_act133,2026-03-04,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[2, 1, 1, 1, 1, 1, 2, 2, 1, 2, 1, 1, 2, 2, 2, ...",...,3,1,254,1,3600.040,2026-03-04 10:26:00,0.512,0,9.28662,1
1,efonseca_ef505_act133,2026-03-03,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 2, 2, 1, 1, 2, 2, 1, 2, 2, 1, 1, 1, 1, ...",...,3,1,244,1,3600.030,2026-03-03 11:47:00,0.468,0,9.26066,1
2,efonseca_ef505_act133,2026-03-02,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1, ...",...,3,1,182,1,3600.030,2026-03-02 11:41:00,0.376,0,9.77846,1
3,efonseca_ef505_act133,2026-02-28,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[2, 1, 2, 1, 1, 1, 2, 1, 2, 2, 1, 2, 2, 1, 1, ...","[2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, ...",...,3,1,83,1,3143.010,2026-02-28 12:06:00,0.196,0,9.74901,1
4,efonseca_ef505_act133,2026-02-27,0,3,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 1, 2, ...",...,2,3,293,38,2960.160,2026-02-27 10:20:00,1.168,0,6.07871,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3513,jyanar_ya071,2026-01-08,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, ...","[1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 2, ...",...,1,<NA>,10,1,203.786,2026-01-08 10:52:00,0.080,0,13.52700,1
3514,jyanar_ya071,2026-01-07,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 1, 2, 1]","[127, 127, 127, 127, 2, 2, 127, 127, 127, 2, 1...",...,1,<NA>,13,1,3762.830,2026-01-07 11:13:00,0.068,0,412.32000,1
3515,jyanar_ya071,2026-01-06,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2]","[127, 127, 127, 127, 127, 127, 127, 127]",...,1,<NA>,8,1,3301.600,2026-01-06 10:53:00,0.000,0,412.77700,1
3516,jyanar_ya071,2026-01-05,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2, 1, 2]","[1, 1, 1, 2, 2, 2, 1, 2, 1, 127]",...,1,<NA>,10,1,1893.440,2026-01-05 11:02:00,0.072,0,171.05800,1


In [119]:
subject_data = wwa.get_subject_data()
subject_data = subject_data.loc[:, ['user_id', 'subject_fullname', 'cage', 'headplate_image_path', 'last_weight', 'water_per_day']]
#subject_data = subject_data.loc[subject_data['user_id'] != 'testuser',:]
subject_data = subject_data.reset_index(drop=True)

dictionary_rename_cols =\
{'user_id': 'owner',
 'subject_fullname': 'subject name',
 'cage': 'cage',
 'headplate_image_path': 'headplate',
 'last_weight': 'last weight',
  'water_per_day': 'water per day',
 }

subject_data = subject_data.rename(columns=dictionary_rename_cols)
subject_data = subject_data[list(dictionary_rename_cols.values())]
subject_data['headplate'] = subject_data['headplate'].astype(str)

headplate_image_paths = subject_data['headplate'].copy()

subject_data['last weight'] = subject_data['last weight'].round(1)
subject_data['water per day'] = subject_data['water per day'].round(1)


subject_data['headplate'] = ''
subject_data['today weight'] = ''
subject_data['today water'] = ''



In [120]:
# Copy the source file to the destination, replacing if it exists
shutil.copy2(WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME_TEMPLATE, WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME)

book = openpyxl.load_workbook(WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME)
sheet = book['Sheet1'] # Replace 'Sheet1' with your sheet name

#sheet.delete_rows(3, sheet.max_row - 1) #



In [121]:
#for r in dataframe_to_rows(subject_data, index=False, header=False):
    #sheet.append(r)


In [122]:
for r_idx, row in enumerate(dataframe_to_rows(subject_data, index=False, header=False), start=3):
    for c_idx, value in enumerate(row, start=1):
        sheet.cell(row=r_idx, column=c_idx, value=value)

In [123]:
column_headplate = subject_data.columns.get_loc('headplate') + 1
column_headplate = openpyxl.utils.get_column_letter(column_headplate)


for i in range(headplate_image_paths.shape[0]):

    headplate_path = headplate_image_paths[i]
    cell_image = column_headplate+str(i+3)

    if pathlib.Path.is_file(pathlib.Path(headplate_path)):
        img = Image(headplate_path)
        img.width = 60
        img.height = 60
        sheet.add_image(img, cell_image)
        sheet.row_dimensions[i+3].height = 50 
    else:
        sheet.row_dimensions[i+3].height = 15

In [124]:
book.save(WEIGHING_GUI_REPLACEMENT_SPREADHSHEET_FILENAME)